In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from PIL import Image
import os
import numpy as np
import plotly.graph_objects as go
from scipy.spatial import cKDTree
from sklearn.decomposition import PCA
import torch.nn as nn

def detect_background_color(img, samples_per_edge=100):
    """
    Safely detect the background color by analyzing image edges.
    Uses a more robust sampling approach that avoids zero-step slicing.
    
    Args:
        img: Input image in BGR format
        samples_per_edge: Number of samples to take from each edge
        
    Returns:
        tuple: RGB color values of detected background
    """
    height, width = img.shape[:2]
    
    # Ensure we take at least one sample per edge
    samples_per_edge = max(1, min(samples_per_edge, min(height, width)))
    
    # Calculate step sizes (ensure non-zero)
    width_step = max(1, width // samples_per_edge)
    height_step = max(1, height // samples_per_edge)
    
    # Sample pixels from edges
    edge_pixels = []
    
    # Sample top and bottom edges
    for x in range(0, width, width_step):
        edge_pixels.append(img[0, x])        # Top edge
        edge_pixels.append(img[-1, x])       # Bottom edge
    
    # Sample left and right edges
    for y in range(0, height, height_step):
        edge_pixels.append(img[y, 0])        # Left edge
        edge_pixels.append(img[y, -1])       # Right edge
    
    # Convert to numpy array
    edge_pixels = np.array(edge_pixels)
    
    # Use K-means clustering to find dominant color
    kmeans = KMeans(n_clusters=3, n_init=10)
    kmeans.fit(edge_pixels)
    
    # Get the most frequent color cluster
    unique, counts = np.unique(kmeans.labels_, return_counts=True)
    dominant_cluster = unique[np.argmax(counts)]
    background_color = kmeans.cluster_centers_[dominant_cluster].astype(int)
    
    # Convert from BGR to RGB
    return tuple(background_color[::-1])

def detect_shadows_enhanced(img, background_color):
    """
    Enhanced shadow detection that preserves subtle lighting transitions and reflections.
    This function analyzes both global and local lighting patterns to identify shadows
    while being careful not to misclassify object details as shadows.
    
    Args:
        img: Input image in BGR format
        background_color: RGB tuple of the background color
    Returns:
        numpy.ndarray: Shadow mask where 1 indicates shadow pixels
    """
    # First, convert background color from RGB to BGR for OpenCV
    bg_color = background_color[::-1]
    
    # Create a background image matching our detected background color
    background = np.full_like(img, bg_color)
    
    # Convert both images to LAB color space for better lighting analysis
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    bg_lab = cv2.cvtColor(background, cv2.COLOR_BGR2LAB)
    
    # Extract L (lightness) channels
    l_img = lab[:,:,0].astype(np.float32)
    l_bg = bg_lab[:,:,0].astype(np.float32)
    
    # Calculate local average of lightness using multiple scales
    shadow_masks = []
    for kernel_size in [(21, 21), (41, 41)]:  # Multiple scales for different shadow sizes
        # Calculate local lightness average
        local_mean = cv2.GaussianBlur(l_img, kernel_size, 0)
        
        # Calculate lightness difference from background
        l_diff = np.abs(l_img - l_bg)
        local_diff = np.abs(local_mean - l_bg)
        
        # Create adaptive threshold based on local contrast
        threshold = np.mean(l_diff) * 0.5 + local_diff * 0.2
        
        # Identify shadow regions
        shadow = (l_diff < threshold) & (l_img < l_bg)
        shadow_masks.append(shadow)
    
    # Combine shadow masks from different scales
    shadow_mask = np.logical_or.reduce(shadow_masks)
    
    # Analyze color differences to prevent misclassifying colored regions as shadows
    a_diff = np.abs(lab[:,:,1] - bg_lab[:,:,1])
    b_diff = np.abs(lab[:,:,2] - bg_lab[:,:,2])
    color_diff = np.sqrt(a_diff**2 + b_diff**2)
    
    # Only keep shadow pixels where color difference is small
    shadow_mask &= (color_diff < 30)
    
    # Clean up the mask
    kernel = np.ones((3,3), np.uint8)
    shadow_mask = cv2.morphologyEx(shadow_mask.astype(np.uint8), 
                                 cv2.MORPH_CLOSE, kernel)
    shadow_mask = cv2.morphologyEx(shadow_mask.astype(np.uint8), 
                                 cv2.MORPH_OPEN, kernel)
    
    return shadow_mask

def refined_background_removal(image_input, background_color=None, edge_smoothing=5, preserve_whites=True):
    """
    Advanced background removal with automatic background color detection and
    improved edge handling.
    
    Args:
        image_input: Path to input image or Image.Image or numpy array
        background_color: RGB tuple or None for auto-detection
        edge_smoothing: Amount of edge smoothing (higher = smoother)
        preserve_whites: Whether to preserve white colors in the object
    """
    def create_color_range_mask(img, target_color, tolerance=20):  # Reduced tolerance
        """Create a more precise mask for colors within tolerance of target color"""
        target_bgr = target_color[::-1]

        # Convert to LAB color space for better color similarity matching
        lab_image = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        lab_target = cv2.cvtColor(np.uint8([[target_bgr]]), cv2.COLOR_BGR2LAB)[0,0]

        # Create bounds with tolerance in LAB space
        lower_bound = np.array([max(0, c - tolerance) for c in lab_target])
        upper_bound = np.array([min(255, c + tolerance) for c in lab_target])

        # Create mask in LAB space
        mask = cv2.inRange(lab_image, lower_bound, upper_bound)

        # Clean up the mask
        kernel = np.ones((3,3), np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

        return mask

    def smooth_edges(mask, smooth_factor):
        """Apply edge smoothing with outline removal"""
        # Convert to float
        mask_float = mask.astype(np.float32) / 255.0
        
        # Apply erosion first to remove thin outlines
        kernel = np.ones((2,2), np.uint8)
        eroded = cv2.erode(mask_float, kernel, iterations=1)
        
        # Multi-scale smoothing
        smoothed = np.zeros_like(mask_float)
        weights_sum = 0
        
        for i in range(1, 4):
            kernel_size = smooth_factor * 2 * i + 1
            current_smooth = cv2.GaussianBlur(eroded, 
                                            (kernel_size, kernel_size), 
                                            0)
            weight = 1.0 / i
            smoothed += current_smooth * weight
            weights_sum += weight
        
        smoothed /= weights_sum
        
        # Threshold the result to make edges cleaner
        smoothed = np.where(smoothed > 0.5, 1.0, 0.0)
        
        return (smoothed * 255).astype(np.uint8)

    def preserve_white_details(img, mask, threshold=250):
        """
        Preserve white details with proper handling of edge cases and division.
        
        Args:
            img: Input image in BGR format
            mask: Binary mask
            threshold: Brightness threshold for white detection
        """
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        
        # Create adaptive threshold with safety checks
        local_mean = cv2.GaussianBlur(gray, (15, 15), 0)
        
        # Avoid division by zero and invalid values
        local_mean = np.clip(local_mean, 1, 254)  # Ensure no zeros or 255s
        adjustment = np.clip((255 - local_mean) * 0.05, 0, threshold)
        local_threshold = threshold - adjustment
        
        # Create white mask with safety checks
        white_areas = gray > local_threshold
        
        # Reduce connectivity to main object
        kernel = np.ones((2,2), np.uint8)
        dilated_mask = cv2.dilate(mask, kernel, iterations=1)
        preserved_whites = white_areas & dilated_mask
        
        # Clean up artifacts
        preserved_whites = cv2.morphologyEx(preserved_whites.astype(np.uint8), 
                                        cv2.MORPH_OPEN, kernel)
        preserved_whites = cv2.morphologyEx(preserved_whites, 
                                        cv2.MORPH_CLOSE, kernel)
        
        return mask | preserved_whites

    def shrink_mask(mask, shrink_percent=1):
        """
        Shrink the mask by a percentage of its dimensions
        Args:
            mask: Binary mask
            shrink_percent: Percentage to shrink (1 = 1%)
        Returns:
            Shrunk mask
        """
        # Get mask dimensions
        height, width = mask.shape[:2]
        
        # Calculate pixels to shrink on each side
        shrink_pixels_y = int(height * (shrink_percent / 100))
        shrink_pixels_x = int(width * (shrink_percent / 100))
        
        # Ensure at least 1 pixel if percentage is too small
        shrink_pixels_y = max(1, shrink_pixels_y)
        shrink_pixels_x = max(1, shrink_pixels_x)
        
        # Create structuring element for erosion
        kernel = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (2 * shrink_pixels_x + 1, 2 * shrink_pixels_y + 1)
        )
        
        # Erode the mask
        shrunk_mask = cv2.erode(mask, kernel, iterations=1)
        
        return shrunk_mask

    # Main processing pipeline
    try:
        print("Loading image...")
        # Input validation and conversion
        if isinstance(image_input, str):
            print("It's a file path - load the image")
            if not os.path.exists(image_input):
                raise ValueError(f"Image file not found: {image_input}")
            image = cv2.imread(image_input)
            if image is None:
                raise ValueError(f"Failed to load image from {image_input}")
        elif isinstance(image_input, np.ndarray):
            print("It's already a numpy array - verify format")
            if len(image_input.shape) != 3 or image_input.shape[2] != 3:
                raise ValueError("Image array must be a 3-channel color image")
            image = image_input
        elif isinstance(image_input, Image.Image):
            
            print("Convert PIL Image to numpy array in BGR format")
            image_array = np.array(image_input)
            image = cv2.cvtColor(image_array, cv2.COLOR_RGB2BGR)
        else:
            raise TypeError("Image input must be either a file path, numpy array, or PIL Image")
        
        print(image.__class__)
        # Handle background color detection
        if background_color is None:
            print("Detecting background color...")
            detected_color = detect_background_color(image)
            print(f"Detected background color (RGB): {detected_color}")
            bg_color = detected_color
        else:
            bg_color = background_color
        
        print("Detecting shadows...")
        shadow_mask = detect_shadows_enhanced(image, bg_color)
        
        print("Creating color-based mask...")
        color_mask = create_color_range_mask(image, bg_color)
        
        # Combine color and shadow masks
        combined_mask = (color_mask | shadow_mask)

        
        
        # Invert mask (we want to keep the object, not the background)
        object_mask = cv2.bitwise_not(combined_mask)
        
        # Preserve white details if requested
        if preserve_whites:
            print("Preserving white details...")
            object_mask = preserve_white_details(image, object_mask)

        
        # Apply edge smoothing
        if edge_smoothing > 0:
            print("Smoothing edges...")
            object_mask = smooth_edges(object_mask, edge_smoothing)

        # Shrinking mask
        print("Shrinking mask...")
        object_mask = shrink_mask(object_mask, shrink_percent=0.5)
        
        
        # Create output images
        alpha = object_mask
        rgba = cv2.cvtColor(image, cv2.COLOR_BGR2BGRA)
        rgba[:, :, 3] = alpha
        
        result = image.copy()
        result[alpha == 0] = [0, 0, 0]
        
        # Convert to RGB for display
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        result_rgb = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)
        
        checkered = np.zeros((image.shape[0], image.shape[1], 3), dtype=np.uint8)
        checkered[::20, ::20] = [200, 200, 200]
        checkered[10::20, 10::20] = [200, 200, 200]
        
        alpha_3d = alpha[:,:,np.newaxis] / 255.0
        blended = (image_rgb * alpha_3d + checkered * (1 - alpha_3d)).astype(np.uint8)
        
        return image_rgb, alpha, result_rgb, blended, rgba
        
    except Exception as e:
        print(f"Error during processing: {str(e)}")
        return None

In [2]:
front_results = refined_background_removal(
            "../static/uploads/Chair/front.jpg",
            background_color=None,  # Enable auto-detection
            edge_smoothing=5,
            preserve_whites=True
        )

front_feature = front_results[2];

back_results = refined_background_removal(
            "../static/uploads/Chair/back.jpg",
            background_color=None,  # Enable auto-detection
            edge_smoothing=5,
            preserve_whites=True
        )

back_feature = back_results[2];
top_results = refined_background_removal(
            "../static/uploads/Chair/top.jpg",
            background_color=None,  # Enable auto-detection
            edge_smoothing=5,
            preserve_whites=True
        )

top_feature = top_results[2];
top_binary_mask = top_results[1];

Loading image...
It's a file path - load the image
<class 'numpy.ndarray'>
Detecting background color...


c:\Users\ACER\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:1474: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (3). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


Detected background color (RGB): (255, 255, 255)
Detecting shadows...
Creating color-based mask...
Preserving white details...
Smoothing edges...
Shrinking mask...
Loading image...
It's a file path - load the image
<class 'numpy.ndarray'>
Detecting background color...
Detected background color (RGB): (255, 255, 255)
Detecting shadows...
Creating color-based mask...
Preserving white details...
Smoothing edges...
Shrinking mask...
Loading image...
It's a file path - load the image
<class 'numpy.ndarray'>
Detecting background color...


c:\Users\ACER\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:1474: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (3). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
c:\Users\ACER\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:1474: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (3). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


Detected background color (RGB): (255, 255, 255)
Detecting shadows...
Creating color-based mask...
Preserving white details...
Smoothing edges...
Shrinking mask...


In [3]:
import torch
import os
import cv2
import numpy as np
from PIL import Image
import plotly.graph_objects as go
from transformers import pipeline
 
# Get depth map using Depth Anything model
device = "cuda" if torch.cuda.is_available() else "cpu"
depth_model = pipeline("depth-estimation", model="depth-anything/Depth-Anything-V2-base-hf", device=device)

In [4]:
def get_depth_map(result_rgb):
    # Use result_rgb directly for depth estimation (it's already in RGB format)
    image_pil = Image.fromarray(result_rgb)
    depth_predictions = depth_model(image_pil)
    depth_map = np.array(depth_predictions["depth"])

    # Process depth map with dimensions from result_rgb
    height, width = result_rgb.shape[:2]
    depth_map_resized = cv2.resize(depth_map, (width, height))

    return depth_map_resized

front_depth_map = get_depth_map(front_feature)
back_depth_map = get_depth_map(back_feature)
top_depth_map = get_depth_map(top_feature)

In [51]:
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KDTree
import plotly.graph_objects as go
import numpy as np
import sklearn
import scipy

In [ ]:
def load_and_normalize_images(image_paths, target_size=256):
    """
    Load and normalize all images to the specified target size while maintaining aspect ratios.
    Uses background color detection for padding.

    The function:
    1. Loads each image
    2. Detects the background color
    3. Resizes while maintaining aspect ratio
    4. Creates a square canvas with the detected background color
    5. Centers the resized image in the canvas
    """
    print("\nLoading and normalizing images...")

    images = {}
    for view_type, path in image_paths.items():
        if path is None:
            continue

        # Load the image
        img = cv2.imread(path)
        if img is None:
            raise ValueError(f"Failed to load image: {path}")

        # Detect background color (returns RGB)
        bg_color_rgb = detect_background_color(img)
        # Convert RGB to BGR for OpenCV
        bg_color_bgr = bg_color_rgb[::-1]
        print(f"{view_type.capitalize()} view background color (RGB): {bg_color_rgb}")

        # Calculate aspect ratio preserving dimensions
        h, w = img.shape[:2]
        aspect = w / h

        if aspect > 1:
            new_w = target_size
            new_h = int(target_size / aspect)
        else:
            new_h = target_size
            new_w = int(target_size * aspect)

        # Resize image
        resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)

        # Create square canvas with detected background color (in BGR)
        square_img = np.full(
            (target_size, target_size, 3), bg_color_bgr, dtype=np.uint8
        )

        # Calculate padding to center the image
        pad_y = (target_size - new_h) // 2
        pad_x = (target_size - new_w) // 2

        # Place resized image in center
        square_img[pad_y : pad_y + new_h, pad_x : pad_x + new_w] = resized

        images[view_type] = square_img
        print(
            f"{view_type.capitalize()} view normalized size: {square_img.shape[1]}x{square_img.shape[0]}"
        )

    return images, target_size


def remove_outliers(
    fig, spatial_strictness=1.0, color_threshold=0.05, min_cluster_size=5
):
    """
    Removes outliers from a 3D point cloud based on both spatial distribution and color.

    Parameters:
    -----------
    fig : plotly.graph_objects.Figure
        Input figure containing the 3D scatter plot
    spatial_strictness : float, default=1.0
        Controls how strict the spatial clustering should be:
        - Lower values (e.g., 0.5) are more lenient and keep more points
        - Higher values (e.g., 2.0) are stricter and remove more points
        - 1.0 is the default balanced setting
    color_threshold : float, default=0.05
        The minimum percentage (0.0 to 1.0) of points a color cluster needs to be considered valid
    min_cluster_size : int, default=5
        Minimum number of points needed to form a valid cluster

    Returns:
    --------
    plotly.graph_objects.Figure
        New figure with outliers removed
    """

    def calculate_dynamic_dbscan_params(points, strictness):
        # Create KD-tree for efficient nearest neighbor search
        point_tree = KDTree(points)
        # Get distances to k nearest neighbors
        distances, _ = point_tree.query(points, k=6)  # k=6 gives 5 neighbors + self

        # Calculate average distance to nearest neighbors
        avg_distance = np.mean(distances[:, 1:])  # Exclude distance to self

        # Adjust eps based on strictness parameter
        # Lower strictness = larger eps = more lenient clustering
        eps = avg_distance * (2.0 / strictness)

        # Calculate dynamic minimum samples
        # Base number is min_cluster_size, scaled by log of point count
        point_count_factor = (
            np.log10(len(points)) / 4
        )  # Divide by 4 to make it less strict
        min_samples = max(min_cluster_size, int(point_count_factor * min_cluster_size))

        return eps, min_samples

    def analyze_color_clusters(colors, threshold):
        """
        Analyzes color distribution using color distance and density-based grouping.

        Parameters:
        -----------
        colors : numpy.ndarray
            Array of RGB colors
        threshold : float
            Threshold for considering a color group significant (0.0 to 1.0)

        Returns:
        --------
        numpy.ndarray
            Boolean mask of valid colors
        """

        def color_distance(c1, c2):
            """
            Calculates a weighted color distance that's more perceptually accurate.
            Gives more weight to differences in the same color channel.
            """
            # Convert to float to avoid overflow
            c1 = c1.astype(float)
            c2 = c2.astype(float)

            # Calculate channel differences
            dr = c1[0] - c2[0]
            dg = c1[1] - c2[1]
            db = c1[2] - c2[2]

            # Weight the differences (giving more weight to same-channel differences)
            return np.sqrt(2 * dr * dr + 4 * dg * dg + 3 * db * db)

        # Find the dominant color (the color that appears most frequently)
        unique_colors, color_counts = np.unique(colors, axis=0, return_counts=True)
        dominant_color = unique_colors[np.argmax(color_counts)]

        # Calculate distances from each point to the dominant color
        distances = np.array(
            [color_distance(color, dominant_color) for color in colors]
        )

        # Calculate adaptive threshold based on the distribution of distances
        distance_threshold = np.percentile(distances, threshold * 100)

        # Create mask for colors within threshold
        valid_colors = distances <= distance_threshold

        return valid_colors

    # Extract points and colors from figure
    trace = fig.data[0]
    points = np.column_stack([trace.x, trace.y, trace.z])
    colors = np.array(
        [list(map(int, c.strip("rgb()").split(","))) for c in trace.marker.color]
    )

    # Scale the points
    scaler = StandardScaler()
    xy_scaled = scaler.fit_transform(points[:, :2])
    z_scaled = scaler.fit_transform(points[:, 2:3]) * 2  # Weight depth more
    points_scaled = np.column_stack([xy_scaled, z_scaled])

    # Get dynamic parameters and perform spatial clustering
    eps, min_samples = calculate_dynamic_dbscan_params(
        points_scaled, spatial_strictness
    )
    spatial_clusters = DBSCAN(eps=eps, min_samples=min_samples, n_jobs=-1).fit_predict(
        points_scaled
    )

    # Get valid points from spatial clustering
    valid_spatial = spatial_clusters != -1

    # Get valid points from color analysis
    valid_colors = analyze_color_clusters(colors, color_threshold)

    # Combine both masks
    valid_points = valid_spatial & valid_colors

    # Print final statistics
    total_points = len(points)
    kept_points = np.sum(valid_points)

    print(f"Outliers removed: {total_points - kept_points}")

    # Create new figure with cleaned points
    cleaned_fig = go.Figure(
        data=[
            go.Scatter3d(
                x=points[valid_points, 0],
                y=points[valid_points, 1],
                z=points[valid_points, 2],
                mode="markers",
                marker=dict(
                    size=trace.marker.size,
                    color=[f"rgb({r},{g},{b})" for r, g, b in colors[valid_points]],
                    opacity=trace.marker.opacity,
                ),
            )
        ]
    )

    # Copy layout from original figure
    cleaned_fig.update_layout(fig.layout)

    return cleaned_fig


# def fill_gaps_with_interpolation(front_points, back_points, num_interpolation_steps=5):
#     """
#     Fills gaps between front and back point clouds using geometric interpolation.

#     This function:
#     1. Identifies corresponding regions between views
#     2. Creates interpolated points to fill gaps
#     3. Maintains smooth transitions in both geometry and color
#     """
#     def find_corresponding_points(source_points, target_points, max_distance):
#         """Finds pairs of points that likely correspond between views."""
#         tree = sklearn.neighbors.KDTree(target_points)
#         distances, indices = tree.query(source_points, k=1)
#         valid_pairs = distances < max_distance
#         return source_points[valid_pairs], target_points[indices[valid_pairs]]

#     def create_interpolated_points(p1, p2, steps):
#         """Creates smooth interpolation between two points."""
#         # Use cubic interpolation for smoother transitions
#         t = np.linspace(0, 1, steps)
#         # Hermite interpolation for position
#         points = []
#         for ti in t:
#             # Cubic interpolation weight
#             h1 = 2*ti**3 - 3*ti**2 + 1
#             h2 = -2*ti**3 + 3*ti**2
#             # Create interpolated point
#             point = h1*p1 + h2*p2
#             points.append(point)
#         return np.array(points)

#     # Parameters for finding corresponding points
#     max_correspondence_distance = np.mean([
#         np.std(front_points, axis=0),
#         np.std(back_points, axis=0)
#     ]) * 0.5

#     # Find corresponding points between views
#     front_corresp, back_corresp = find_corresponding_points(
#         front_points, back_points, max_correspondence_distance)

#     # Generate interpolated points between corresponding pairs
#     interpolated_points = []
#     for fp, bp in zip(front_corresp, back_corresp):
#         new_points = create_interpolated_points(fp, bp, num_interpolation_steps)
#         interpolated_points.extend(new_points)


#     return np.array(interpolated_points)
def create_object_mesh(image_input, depth_threshold=0.0, target_size=256):
    """
    Creates a 3D mesh from either an image path or a numpy array using refined background removal.
    """
    # Input validation and conversion
    if isinstance(image_input, str):
        # Normalize the image first using our original normalization function
        images, _ = load_and_normalize_images({"input": image_input}, target_size)
        image = images["input"]
    elif isinstance(image_input, np.ndarray):
        image = image_input
    elif isinstance(image_input, Image.Image):
        image_array = np.array(image_input)
        image = cv2.cvtColor(image_array, cv2.COLOR_RGB2BGR)
    else:
        raise TypeError(
            "Image input must be either a file path, numpy array, or PIL Image"
        )

    try:

        # After background removal
        _, alpha_mask, result_rgb, _, _ = refined_background_removal(image)
        print(f"\nAfter background removal:")
        print(f"Result RGB shape: {result_rgb.shape}")
        print(f"Alpha mask shape: {alpha_mask.shape}")
        print(f"Non-zero alpha pixels: {np.count_nonzero(alpha_mask)}")

        # Get depth map
        device = "cuda" if torch.cuda.is_available() else "cpu"
        depth_model = pipeline(
            "depth-estimation",
            model="depth-anything/Depth-Anything-V2-base-hf",
            device=device,
        )
        depth_predictions = depth_model(Image.fromarray(result_rgb))
        depth_map = np.array(depth_predictions["depth"])

        print(f"\nDepth map statistics:")
        print(f"Depth map shape: {depth_map.shape}")
        print(f"Depth range: {depth_map.min():.3f} to {depth_map.max():.3f}")

        # Process depth map
        height, width = result_rgb.shape[:2]
        depth_map_resized = cv2.resize(depth_map, (width, height))
        alpha_mask_resized = cv2.resize(
            alpha_mask, (depth_map_resized.shape[1], depth_map_resized.shape[0])
        )
        depth_mask = alpha_mask_resized > 0

        print(f"\nAfter resizing:")
        print(f"Resized depth map shape: {depth_map_resized.shape}")
        print(f"Valid mask points: {np.count_nonzero(depth_mask)}")

        # Normalize depth map
        depth_norm = (depth_map_resized - depth_map_resized.min()) / (
            depth_map_resized.max() - depth_map_resized.min()
        )
        print(
            f"\nNormalized depth range: {depth_norm.min():.3f} to {depth_norm.max():.3f}"
        )

        # Create points
        x_coords, y_coords = np.meshgrid(np.arange(width), np.arange(height))
        valid_points = depth_mask & (depth_norm > depth_threshold)

        x_filtered = x_coords[valid_points].tolist()
        y_filtered = y_coords[valid_points].tolist()
        z_filtered = depth_norm[valid_points].tolist()
        colors_filtered = result_rgb[valid_points].tolist()

        print(f"\nPoint generation results:")
        print(f"Total possible points: {width * height}")
        print(f"Points after masking: {np.count_nonzero(depth_mask)}")
        print(f"Final points after depth threshold: {len(x_filtered)}")

        # Create the figure and return
        fig = go.Figure(
            data=[
                go.Scatter3d(
                    x=x_filtered,
                    y=y_filtered,
                    z=z_filtered,
                    mode="markers",
                    marker=dict(
                        size=2,
                        color=[f"rgb({r},{g},{b})" for r, g, b in colors_filtered],
                        opacity=1,
                    ),
                )
            ]
        )

        print("\n=== Mesh Creation Complete ===")
        return fig

    except Exception as e:
        print(f"Error during mesh creation: {str(e)}")
        return None


def combine_views_with_gap_filling(combined_fig, point_spacing=5):
    """
    Creates a complete fill from surface to center with configurable point spacing.
    
    Parameters:
    -----------
    combined_fig : plotly.graph_objects.Figure
        The combined mesh containing front and back views
    point_spacing : float, default=5
        Distance between consecutive points during filling.
        Higher values create more space between points.
    """
    def create_fade_color(original_color, fade_factor):
        """Creates a smoothly faded color with gamma correction for natural appearance."""
        original_color = np.array(original_color, dtype=float)
        fade_factor = np.clip(fade_factor, 0, 1)
        
        # Apply gamma correction for natural color fading
        gamma = 2.2
        color_linear = (original_color / 255.0) ** gamma
        faded_linear = color_linear * fade_factor
        faded_color = 255.0 * (faded_linear ** (1.0/gamma))
        
        return np.clip(faded_color, 0, 255).astype(np.uint8)

    def fill_from_surface(surface_points, surface_colors, is_front):
        """
        Creates filling points from surface to center with dynamic point count.
        Points are spaced according to point_spacing parameter.
        """
        filled_points = []
        filled_colors = []
        
        for point, color in zip(surface_points, surface_colors):
            current_z = point[2]
            target_z = 0.5  # Center point
            
            # Calculate distance to center
            z_distance = abs(target_z - current_z)
            
            # Calculate number of points needed based on distance and spacing
            # Add 1 to ensure we always have at least one point
            num_points = max(2, int(z_distance / (point_spacing * 0.01)) + 1)
            
            # Create sequence of points
            z_values = np.linspace(current_z, target_z, num_points)
            
            # For each point along the path to center
            for i, z in enumerate(z_values):
                # Calculate progress (0 at surface, 1 at center)
                progress = i / (num_points - 1)
                
                # Create new point
                new_point = point.copy()
                new_point[2] = z
                
                # Move x coordinate toward center with smooth transition
                if is_front:
                    # Front view points move backward toward center
                    new_point[0] = point[0] - progress * (point[0] - split_x)
                else:
                    # Back view points move forward toward center
                    new_point[0] = point[0] + progress * (split_x - point[0])
                
                # Calculate color fade based on distance from surface
                fade_factor = 1.0 - progress
                faded_color = create_fade_color(color, fade_factor)
                
                filled_points.append(new_point)
                filled_colors.append(faded_color)
        
        if filled_points:
            return np.array(filled_points), np.array(filled_colors)
        return [], []

    # Extract points and colors
    points = np.column_stack([
        combined_fig.data[0].x,
        combined_fig.data[0].y,
        combined_fig.data[0].z
    ])
    colors = np.array([list(map(int, c.strip('rgb()').split(','))) 
                      for c in combined_fig.data[0].marker.color])
    
    # Find the center splitting plane
    split_x = np.median(points[:, 0])
    
    # Separate into front and back views
    front_mask = points[:, 0] > split_x
    back_mask = ~front_mask
    
    print("\nSurface Analysis:")
    print(f"Front surface points: {np.sum(front_mask)}")
    print(f"Back surface points: {np.sum(back_mask)}")
    
    # Fill from both surfaces toward center
    front_filled_points, front_filled_colors = fill_from_surface(
        points[front_mask], colors[front_mask], True)
    back_filled_points, back_filled_colors = fill_from_surface(
        points[back_mask], colors[back_mask], False)
    
    print("\nFill Results:")
    print(f"Added {len(front_filled_points)} front fill points")
    print(f"Added {len(back_filled_points)} back fill points")
    
    # Combine all points
    all_points = np.vstack([
        points,  # Original surface points
        front_filled_points,
        back_filled_points
    ])
    
    all_colors = np.vstack([
        colors,  # Original surface colors
        front_filled_colors,
        back_filled_colors
    ])
    
    # Create visualization with updated point size
    filled_fig = go.Figure(data=[go.Scatter3d(
        x=all_points[:, 0],
        y=all_points[:, 1],
        z=all_points[:, 2],
        mode='markers',
        marker=dict(
            size=5,  # Fixed point size of 5
            color=[f'rgb({r},{g},{b})' for r,g,b in all_colors],
            opacity=combined_fig.data[0].marker.opacity
        )
    )])
    
    # Copy layout settings
    filled_fig.update_layout(combined_fig.layout)
    
    return filled_fig


def combine_front_back_meshes(
    front_fig, back_fig, depth_threshold=0.25, sampling_rate=1.0
):
    """Combines front and back meshes with detailed debugging information."""
    print("\n=== Starting Mesh Combination ===")

    def extract_points(fig, view_name):
        if fig is None or not fig.data:
            print(f"No data found for {view_name} view")
            return [], [], [], []

        trace = fig.data[0]
        points = (
            np.array(trace.x),
            np.array(trace.y),
            np.array(trace.z),
            [c.strip("rgb()").split(",") for c in trace.marker.color],
        )
        print(f"\n{view_name} view initial points: {len(points[0])}")
        return points

    def sample_points(x, y, z, colors, rate, view_name):
        n_points = len(x)
        n_sample = int(n_points * rate)
        print(f"\nSampling {view_name} view:")
        print(f"Original points: {n_points}")
        print(f"Target points: {n_sample}")

        if n_sample >= n_points:
            return x, y, z, colors

        indices = np.random.choice(n_points, n_sample, replace=False)
        indices.sort()

        sampled = (x[indices], y[indices], z[indices], [colors[i] for i in indices])
        print(f"Points after sampling: {len(sampled[0])}")
        return sampled

    # Extract points
    front_x, front_y, front_z, front_colors = extract_points(front_fig, "Front")
    back_x, back_y, back_z, back_colors = extract_points(back_fig, "Back")

    if len(front_x) == 0 and len(back_x) == 0:
        print("No points found in either view")
        return None

    # Sample points if needed
    front_x, front_y, front_z, front_colors = sample_points(
        front_x, front_y, front_z, front_colors, sampling_rate, "Front"
    )
    back_x, back_y, back_z, back_colors = sample_points(
        back_x, back_y, back_z, back_colors, sampling_rate, "Back"
    )

    # Process dimensions
    original_width = max(
        np.max(front_x) if len(front_x) > 0 else 0,
        np.max(back_x) if len(back_x) > 0 else 0,
    )
    original_height = max(
        np.max(front_y) if len(front_y) > 0 else 0,
        np.max(back_y) if len(back_y) > 0 else 0,
    )

    print(f"\nDimensions:")
    print(f"Original width: {original_width}")
    print(f"Original height: {original_height}")
    print(
        f"Aspect ratio: {original_width/original_height if original_height != 0 else 'N/A'}"
    )

    # Normalize and combine points
    print("\nNormalizing depths:")
    for view_name, points in [
        ("Front", (front_x, front_y, front_z)),
        ("Back", (back_x, back_y, back_z)),
    ]:
        if len(points[0]) > 0:
            print(
                f"{view_name} z-range: {points[2].min():.3f} to {points[2].max():.3f}"
            )

    # Center and normalize points
    if len(front_x) > 0:
        front_width = np.max(front_x) - np.min(front_x)
        front_x_centered = front_x - np.min(front_x) - front_width / 2
        front_z_norm = 0.5 + (front_z - np.min(front_z)) * depth_threshold / (
            np.max(front_z) - np.min(front_z)
        )

    if len(back_x) > 0:
        back_width = np.max(back_x) - np.min(back_x)
        back_x_centered = -(back_x - np.min(back_x) - back_width / 2)
        back_z_norm = 0.5 - (back_z - np.min(back_z)) * depth_threshold / (
            np.max(back_z) - np.min(back_z)
        )

    # Combine points
    combined_x = np.concatenate([front_x_centered, back_x_centered])
    combined_y = np.concatenate([front_y, back_y])
    combined_z = np.concatenate([front_z_norm, back_z_norm])
    combined_colors = [f'rgb({",".join(c)})' for c in front_colors] + [
        f'rgb({",".join(c)})' for c in back_colors
    ]

    print(f"\nFinal combined point count: {len(combined_x)}")
    print(f"Combined z-range: {combined_z.min():.3f} to {combined_z.max():.3f}")

    # Create combined figure
    combined_fig = go.Figure(
        data=[
            go.Scatter3d(
                x=combined_x,
                y=combined_y,
                z=combined_z,
                mode="markers",
                marker=dict(size=10, color=combined_colors, opacity=1),
                showlegend=False,
            )
        ]
    )

    # Update layout
    combined_fig.update_layout(
        scene=dict(
            aspectratio=dict(x=1, y=1 / (original_width / original_height), z=0.5),
            aspectmode="manual",
            camera=dict(
                eye=dict(x=1.25, y=-0.25, z=1.25),
                up=dict(x=0, y=0, z=0),
                center=dict(x=0, y=0, z=0),
            ),
            xaxis=dict(
                range=[-original_width / 2, original_width / 2],
                showticklabels=False,
                showgrid=False,
                zeroline=False,
                showline=False,
                showbackground=False,
            ),
            yaxis=dict(
                range=[0, original_height],
                showticklabels=False,
                showgrid=False,
                zeroline=False,
                showline=False,
                showbackground=False,
            ),
            zaxis=dict(
                range=[0, 1],
                showticklabels=False,
                showgrid=False,
                zeroline=False,
                showline=False,
                showbackground=False,
            ),
        ),
        uirevision="true",
        showlegend=False,
        margin=dict(l=0, r=0, t=0, b=0, pad=0),
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
    )

    print("\n=== Mesh Combination Complete ===")
    return combined_fig


def process_orthogonal_views(
    front,
    back=None,
    top=None,
    target_size=256,
    spatial_strictness=0.5,
    color_threshold=0.99,
):
    """
    Process orthogonal views with normalized image sizes.
    """
    # Process all three views
    image_paths = {"front": front, "back": back, "top": top}

    try:
        # First normalize all images
        normalized_images, _ = load_and_normalize_images(
            image_paths, target_size=target_size
        )

        # Create meshes for each view
        front_fig = create_object_mesh(
            normalized_images["front"], depth_threshold=0.1, target_size=target_size
        )
        front_fig = remove_outliers(
            front_fig,
            spatial_strictness=spatial_strictness,
            color_threshold=color_threshold,
        )

        back_fig = None
        if back is not None:
            back_fig = create_object_mesh(
                normalized_images["back"], depth_threshold=0.1, target_size=target_size
            )
            back_fig = remove_outliers(
                back_fig,
                spatial_strictness=spatial_strictness,
                color_threshold=color_threshold,
            )

        # top_fig = None
        # if back is not None:
        #     top_fig = create_object_mesh(normalized_images['top'],  target_size=target_size)
        #     top_fig = remove_outliers(top_fig, spatial_strictness=0.5, color_threshold=0.05)

        # Combine meshes
        combined_mesh = combine_front_back_meshes(front_fig, back_fig)
        # combined_mesh.show()
        filled_mesh = combine_views_with_gap_filling(
            combined_mesh
        )
        # filled_mesh.show()
        return filled_mesh.to_dict()

    except Exception as e:
        print(f"Error during processing: {str(e)}")
        return None


orthogonal_views = process_orthogonal_views(
    front="../static/uploads/Chair/front.jpg",
    back="../static/uploads/Chair/back.jpg",
    top="../static/uploads/Chair/top.jpg",
    target_size=256,
)


Loading and normalizing images...
Front view background color (RGB): (255, 255, 255)
Front view normalized size: 256x256
Back view background color (RGB): (255, 255, 255)
Back view normalized size: 256x256
Top view background color (RGB): (255, 255, 255)


c:\Users\ACER\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:1474: ConvergenceWarning:

Number of distinct clusters (1) found smaller than n_clusters (3). Possibly due to duplicate points in X.

c:\Users\ACER\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:1474: ConvergenceWarning:

Number of distinct clusters (1) found smaller than n_clusters (3). Possibly due to duplicate points in X.

c:\Users\ACER\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:1474: ConvergenceWarning:

Number of distinct clusters (1) found smaller than n_clusters (3). Possibly due to duplicate points in X.



Top view normalized size: 256x256
Loading image...
It's already a numpy array - verify format
<class 'numpy.ndarray'>
Detecting background color...
Detected background color (RGB): (255, 255, 255)
Detecting shadows...
Creating color-based mask...
Preserving white details...
Smoothing edges...
Shrinking mask...

After background removal:
Result RGB shape: (256, 256, 3)
Alpha mask shape: (256, 256)
Non-zero alpha pixels: 5191


c:\Users\ACER\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:1474: ConvergenceWarning:

Number of distinct clusters (1) found smaller than n_clusters (3). Possibly due to duplicate points in X.




Depth map statistics:
Depth map shape: (256, 256)
Depth range: 10.000 to 255.000

After resizing:
Resized depth map shape: (256, 256)
Valid mask points: 5191

Normalized depth range: 0.000 to 1.000

Point generation results:
Total possible points: 65536
Points after masking: 5191
Final points after depth threshold: 5155

=== Mesh Creation Complete ===
Outliers removed: 61
Loading image...
It's already a numpy array - verify format
<class 'numpy.ndarray'>
Detecting background color...
Detected background color (RGB): (255, 255, 255)
Detecting shadows...
Creating color-based mask...
Preserving white details...
Smoothing edges...
Shrinking mask...

After background removal:
Result RGB shape: (256, 256, 3)
Alpha mask shape: (256, 256)
Non-zero alpha pixels: 5045


c:\Users\ACER\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:1474: ConvergenceWarning:

Number of distinct clusters (1) found smaller than n_clusters (3). Possibly due to duplicate points in X.




Depth map statistics:
Depth map shape: (256, 256)
Depth range: 19.000 to 255.000

After resizing:
Resized depth map shape: (256, 256)
Valid mask points: 5045

Normalized depth range: 0.000 to 1.000

Point generation results:
Total possible points: 65536
Points after masking: 5045
Final points after depth threshold: 4898

=== Mesh Creation Complete ===
Outliers removed: 85

=== Starting Mesh Combination ===

Front view initial points: 5094

Back view initial points: 4813

Sampling Front view:
Original points: 5094
Target points: 5094

Sampling Back view:
Original points: 4813
Target points: 4813

Dimensions:
Original width: 162.0
Original height: 176.0
Aspect ratio: 0.9204545454545454

Normalizing depths:
Front z-range: 0.102 to 0.400
Back z-range: 0.102 to 0.360

Final combined point count: 9907
Combined z-range: 0.250 to 0.750

=== Mesh Combination Complete ===

Surface Analysis:
Front surface points: 4893
Back surface points: 5014

Fill Results:
Added 16539 front fill points
Added 1